# Lab 4: Job Search Agent (Auto-Graded)

You're going to build an agent that:
1. **Reads** about you from a file
2. **Reads** your resume from a file
3. **Searches** the web for relevant job postings
4. **Fetches** actual job descriptions
5. **Recommends** jobs that match your profile

This builds on Lab 3 — same auto-grading, but now you'll **create your own tools**.

---

## The Grading Rubric

Your agent will be asked to find jobs. To pass, it must:

| Requirement | Why |
|-------------|-----|
| Call `read_about_me` | The agent must know who you are and what you're looking for |
| Call `read_resume` | The agent must know your background and skills |
| Call `web_search` | The agent must actually search for jobs online |
| Call `web_fetch` | The agent must read actual job postings |

**All four must happen.** If any is missing, you fail.

- ✅ **Pass = 100 points**
- ❌ **Fail = 0 points**

---

## Setup

Run these cells to get started.

In [1]:
%pip install -r requirements.txt

Looking in indexes: https://pypi.org/simple, https://pypi.fury.io/ericmichael/
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")

True

In [3]:
from agents import Agent
from omniagents import Runner, function_tool
from omniagents.builtin.tools import web_search, web_fetch, write_file, read_file
from omniagents.notebook import evaluate
from markitdown import MarkItDown

---

## Part 1: Create Your Tools

In Lab 3, you used pre-built tools. Now you'll **create your own**.

A tool is just a Python function with the `@function_tool` decorator.

**Your job:** Complete the two tools below.

In [4]:
@function_tool
def read_about_me() -> str:
    """Read information about the user — who they are, what they're looking for, and their preferences."""
    # YOUR TURN: Read and return the contents of examples/lab4/about_me.md
    with open("examples/lab4/about_me.md") as f:
        return f.read()


@function_tool
def read_resume() -> str:
    """Read the user's resume from a docx document — their education, experience, and skills."""
    # YOUR TURN: Convert examples/lab4/resume.docx to markdown and return its contents
    # Use MarkItDown to convert the DOCX document
    md = MarkItDown()
    result = md.convert("examples/lab4/resume.docx")
    return result.text_content

**Hints:**

To read a markdown file:
```python
with open("path/to/file.md") as f:
    return f.read()
```

To convert a Word document to markdown:
```python
md = MarkItDown()
result = md.convert("path/to/file.docx")
return result.text_content
```

---

## Part 2: Write the Instructions

Now tell the agent how to use these tools to find relevant jobs.

Think about:
- How should the agent learn about you? (about me + resume)
- How should it search for jobs?
- How should it evaluate if a job is a good fit?
- What should it return to you?

In [5]:
INSTRUCTIONS = """
You are a helpful job search assistant.

When the user asks you to find jobs:

1. First use your tool read_about_me to learn about the user and their preferences.

2. Then use your tool read_resume to learn about the user's experience, education, and skills.

3. Next use your tools web_search and web_fetch to find job listings that match the user's preferences and qualifications.

4. Finally, return a list of the top 5 job listings that are the best match for the user, along with a brief explanation of why each job is a good fit.

5.once you are done make a job_posting.md file that contains the job listings you found and your explanations for why they are a good fit, using the write_file tool. in here only write one job listing.

6. inside the job_posting.md make some comments for a interviewer to ask a interviewee about the job listing, these comments should be based on the job listing and the user's resume and preferences.
"""

---

## Part 3: Create the Agent

Create the agent with:
- A name
- Your instructions
- **All four tools** (your two custom tools + `web_search` + `web_fetch`)
- A model (use `gpt-5.2`)

In [6]:
recruiter = Agent(
    name="JobFinder",  # YOUR TURN: Give it a name
    instructions=INSTRUCTIONS,
    tools=[read_about_me, read_resume, web_search, web_fetch, write_file],  # YOUR TURN: What tools does it need? (hint: 4 tools)
    model="gpt-5.2",
)

---

## Part 4: Test It!

Run your agent and watch what it does. Does it read your files? Search the web? Fetch job postings?

In [7]:
runner = Runner.from_agent(recruiter)
runner.run_notebook(input="Find me some jobs that would be a good fit for me.")

---

## Part 5: Check Your Grade

Now run the auto-grader. **Green = pass. Red = fail.**

In [9]:
from agents import Agent
from omniagents import Runner, function_tool
from markitdown import MarkItDown
@function_tool
def read_job_posting() -> str:
    """Read the job posting — the role, requirements, and responsibilities."""
    # YOUR TURN: Read and return the contents of examples/lab5/job_posting.md
    
    with open("job_posting.md") as f:
        return f.read()

@function_tool
def read_about_me() -> str:
    """Read information about the user — who they are, what they're looking for, and their preferences."""
    # YOUR TURN: Read and return the contents of examples/lab4/about_me.md
    with open("examples/lab4/about_me.md") as f:
        return f.read()


@function_tool
def read_resume() -> str:
    """Read the user's resume from a docx document — their education, experience, and skills."""
    # YOUR TURN: Convert examples/lab4/resume.docx to markdown and return its contents
    # Use MarkItDown to convert the DOCX document
    md = MarkItDown()
    result = md.convert("examples/lab4/resume.docx")
    return result.text_content

INSTRUCTIONS = """
You are a hiring manager conducting a job interview for a tech company.

## Use the job_posting.md file to understand the role you're hiring for, the requirements, and responsibilities.

## Identity / Personality
You are Sarah Chen, Senior Engineering Manager at the company from the job_posting.md file. You've been with the company 
for 4 years and have hired over 20 engineers. You're known for being warm but thorough — 
you put candidates at ease while still asking substantive questions. You genuinely enjoy 
meeting new people and learning about their experiences.

## Goal
Conduct a realistic 15-20 minute job interview to assess if the candidate is a good fit 
for the role. You want to understand their technical skills, problem-solving ability, 
and whether they'd thrive on your team.

## Starting Context
Before asking ANY questions, you MUST:
1. Use read_job_posting to understand the role you're hiring for
2. Use read_resume to review the candidate's background
3. Use read_about_me to understand what they're looking for
4. Use web_search and web_fetch as needed to fill in any gaps in your understanding of their background or the role

Only after reading all three should you begin the interview.

## Guidance
- Start with a warm introduction — say your name, your role, and give a brief overview of the interview
- Ask 4-5 questions total, mixing:
  - Technical questions related to the job requirements
  - Behavioral questions ("Tell me about a time when...")
  - Questions that reference SPECIFIC items from their resume
- Listen actively — ask follow-up questions based on their answers
- If an answer is vague, probe deeper ("Can you give me a specific example?")
- End by asking if they have questions for you, then explain next steps

## Response Style
- Conversational and warm, but professional
- Keep responses concise — this is a dialogue, not a monologue
- Use the candidate's name occasionally
- React naturally to their answers ("That's interesting!", "I see", etc.)
"""

interviewer = Agent(
    name="Interviewer",  # YOUR TURN: Give it a name
    instructions=INSTRUCTIONS,
    tools=[read_job_posting, read_about_me, read_resume],  # YOUR TURN: What tools does it need? (hint: 3 tools)
    model="gpt-5.2",
)

runner = Runner.from_agent(interviewer)
runner.run_notebook(input="Hi! I'm ready for my interview.")

---

## Part 6: Didn't Pass? Iterate!

**Green = pass. Red = fail.** The grader shows you exactly which tools were missing.

### Common Issues

**"read_about_me was not called"** or **"read_resume was not called"**
- Did you implement the tool functions? (They can't just return `pass`)
- Did you tell the agent to use them in your instructions?
- Did you include them in the `tools=[]` list?

**"web_search was not called"**
- Your instructions didn't tell the agent to search for jobs online

**"web_fetch was not called"**
- Your instructions didn't tell the agent to fetch actual job postings

### The Fix

1. Check **Part 1** — are your tools implemented?
2. Check **Part 2** — do your instructions mention all four steps?
3. Check **Part 3** — did you include all four tools?
4. Re-run **Part 4** to test
5. Re-run **Part 5** to check your grade
6. Repeat until it's green

---

## Part 7: Make It Yours

Once you pass with the example files, **edit the files with your own content**:

1. Open `examples/lab4/about_me.md` — fill in your real info
2. Replace `examples/lab4/resume.docx` with your real resume
3. Re-run the agent and see what jobs it finds for *you*!

---

## Challenge (Optional)

Already passing? Try this: change your model from `gpt-5.2` to `gpt-4.1` (a weaker model).

Can you get it to pass with the same instructions? Or do you need to be even more explicit?

---

## What You Just Learned

You leveled up from Lab 3:

| Lab 3 | Lab 4 |
|-------|-------|
| Used pre-built tools | **Created your own tools** |
| 3 tools to coordinate | 4 tools to coordinate |
| Same evaluation pattern | Same evaluation pattern |

The pattern is always the same:
1. Give the agent tools
2. Write instructions that describe the workflow
3. Test that it actually uses the tools

**Tools are just functions.** If you can write a Python function, you can give your agent new capabilities.